Preparations:
- load prediction.csv
    - drop the lines with the non-used ifs
    - if there is more then one if, take mean

- add column "sample" to master table
- join them
- drop unnecessary columns (matadata, paths, ranks)

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import ast
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score, precision_recall_curve, confusion_matrix

# Configuration
CSV_PATH =  "/home/bri/pymol/data/predictions/preds_ger_orig.csv" 
MASTER_PATH =  "/home/bri/pymol/data/germinal/table/master_table_germinal.csv" 
YAML_PATH = "/home/bri/pymol/EDA/static/metric_directions.yaml"
OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/germinal_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


KEEP_INTERFACES = {"A,C", "B,C"}   # as option if there is more than one, default A,B
palette = {"binder": "#2196F3", "ala_scan": "#FF5722"}


# Load data
df_if = pd.read_csv(CSV_PATH)


#### snir

In [79]:
df_if = preds_df[preds_df["interface"].str.strip().isin(KEEP_INTERFACES)].copy()
print(f"Loaded {len(df_if)} rows. Interfaces: {df_if['interface'].unique()}")

df_if.head(10)


Loaded 70 rows. Interfaces: <StringArray>
['A,C', 'B,C']
Length: 2, dtype: str


,sample,binding,interface,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,...,boltz_template_plddt,boltz_template_iplddt,boltz_template_pde,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_model_path,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis
1,5ig7_abc_cut,False,"A,C",2.572969,95.599717,10.97,1.38,2.321875,94.630252,0.780104,...,0.987334,0.987299,0.258769,0.284754,0.948463,/scicore/home/schwede/barta0000/scoring_pipeli...,0.921477,0.2352,0.9699,0.8588
2,5ig7_abc_cut,False,"B,C",2.572969,95.599717,5.13,0.67,1.720000,95.189670,0.790385,...,0.987334,0.987299,0.258769,0.284754,0.956973,/scicore/home/schwede/barta0000/scoring_pipeli...,0.928806,0.2332,0.9688,0.8586
4,5ig7_abc_cut_A_Y66A,False,"A,C",2.572738,95.767986,17.48,2.33,2.219231,94.959764,0.779923,...,0.986631,0.985549,0.257634,0.291466,0.935394,/scicore/home/schwede/barta0000/scoring_pipeli...,0.898429,0.2315,0.9617,0.8487
5,5ig7_abc_cut_A_Y66A,False,"B,C",2.572738,95.767986,0.73,0.10,1.700000,95.167570,0.792646,...,0.986631,0.985549,0.257634,0.291466,0.945723,/scicore/home/schwede/barta0000/scoring_pipeli...,0.907364,0.2149,0.9585,0.8409
7,5ifj_abc_cut_A_S64A,False,"A,C",2.539446,95.613237,23.78,2.92,2.434375,94.266763,0.775252,...,0.984658,0.983850,0.258604,0.295100,0.932908,/scicore/home/schwede/barta0000/scoring_pipeli...,0.892828,0.2290,0.9581,0.8494
8,5ifj_abc_cut_A_S64A,False,"B,C",2.539446,95.613237,5.41,0.70,1.710000,95.361092,0.782671,...,0.984658,0.983850,0.258604,0.295100,0.944936,/scicore/home/schwede/barta0000/scoring_pipeli...,0.904881,0.2280,0.9569,0.8415
10,5ifj_abc_cut_all_A,False,"A,C",3.597543,93.235998,52.47,5.49,2.602632,84.615603,0.216341,...,0.953720,0.922320,0.256769,0.294560,0.943772,/scicore/home/schwede/barta0000/scoring_pipeli...,0.912904,0.1090,0.6813,0.8530
11,5ifj_abc_cut_all_A,False,"B,C",3.597543,93.235998,16.68,4.56,1.806250,82.884986,0.162327,...,0.953720,0.922320,0.256769,0.294560,0.926717,/scicore/home/schwede/barta0000/scoring_pipeli...,0.872455,0.1163,0.6247,0.8453
13,5ig7_abc_cut_B_G114A,False,"A,C",2.436673,95.869087,24.31,3.01,2.137500,95.580074,0.803864,...,0.986487,0.986081,0.256536,0.278153,0.950070,/scicore/home/schwede/barta0000/scoring_pipeli...,0.924004,0.2335,0.9680,0.8583
14,5ig7_abc_cut_B_G114A,False,"B,C",2.436673,95.869087,0.09,0.01,1.631250,95.483511,0.816920,...,0.986487,0.986081,0.256536,0.278153,0.955974,/scicore/home/schwede/barta0000/scoring_pipeli...,0.927049,0.2016,0.9601,0.8512


#### all

In [88]:
num_cols = df_if.select_dtypes(include=['number']).columns
other_cols = [c for c in df_if.columns if c not in num_cols and c != 'sample']

agg_dict = {col: 'mean' for col in num_cols}
agg_dict.update({col: 'first' for col in other_cols})

df_1if = df_if.groupby('sample').agg(agg_dict).reset_index()

df_1if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,binding,interface,chai_unconstrained_model_path,chai_constrained_model_path,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm,boltz_free_model_path,boltz_template_model_path
0,nb_s103722_mpnn_4_IL20,5.283784,87.857839,116.37,5.25,3.680952,84.373753,0.601250,0.3537,0.5094,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.8735777139663696, 0.8897836804389954]]","[[[0.8735777139663696, 0.2801356613636017], [0...","[[0.8587068915367126, 0.887911856174469]]","[[[0.8587068915367126, 0.29343220591545105], [...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...
1,nb_s105046_mpnn_2_IL3,4.861342,89.483418,63.02,2.94,3.526136,85.492289,0.753162,0.4026,0.5049,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.9269569516181946, 0.3172776699066162]]","[[[0.9269569516181946, 0.16429194808006287], [...","[[0.9257946014404297, 0.36133983731269836]]","[[[0.9257946014404297, 0.2954653203487396], [0...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...
2,nb_s105306_mpnn_2_BHRF,5.242583,89.769324,57.36,2.65,3.632222,88.801675,0.700666,0.4583,0.6721,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.9164183139801025, 0.9224730134010315]]","[[[0.9164183139801025, 0.5580517649650574], [0...","[[0.9157121777534485, 0.922823429107666]]","[[[0.9157121777534485, 0.5435625910758972], [0...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...
3,nb_s105306_mpnn_3_BHRF,12.129656,78.995566,204.71,9.87,6.875641,61.211013,0.036913,0.0798,0.0185,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.9163332581520081, 0.9181700944900513]]","[[[0.9163332581520081, 0.5410971641540527], [0...","[[0.9205971956253052, 0.9215781092643738]]","[[[0.9205971956253052, 0.7132046222686768], [0...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...
4,nb_s109706_mpnn_2_BHRF,5.557438,89.234917,31.00,1.86,3.048485,90.788485,0.738247,0.4322,0.7636,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.9237253665924072, 0.9229375720024109]]","[[[0.9237253665924072, 0.8435201644897461], [0...","[[0.9260150790214539, 0.9301165342330933]]","[[[0.9260150790214539, 0.8554625511169434], [0...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,nb_s949322_mpnn_4_BHRF,5.784341,88.236601,185.07,8.82,3.860000,86.359230,0.628557,0.4430,0.6266,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.8663344979286194, 0.9087797403335571]]","[[[0.8663344979286194, 0.255534827709198], [0....","[[0.8703155517578125, 0.9116320013999939]]","[[[0.8703155517578125, 0.48192840814590454], [...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...
236,nb_s972445_mpnn_2_BHRF,5.676189,87.873886,86.90,4.21,3.539535,87.190624,0.715671,0.4281,0.5489,...,False,"A,B",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/schwede/barta0000/germinal_ds/te...,"[[0.9258394241333008, 0.9077953100204468]]","[[[0.9258394241333008, 0.7158982157707214], [0...","[[0.9366505742073059, 0.9215488433837891]]","[[[0.9366505742073059, 0.8184788227081299], [0...",/scicore/home/schwede/barta0000/germinal_ds/te...,/scicore/home/s

In [89]:
df_2if = df_1if.drop(columns=["binding", "interface","chai_unconstrained_model_path", "chai_constrained_model_path", "boltz_free_model_path", "boltz_template_model_path"])
df_2if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm
0,nb_s103722_mpnn_4_IL20,5.283784,87.857839,116.37,5.25,3.680952,84.373753,0.601250,0.3537,0.5094,...,4.404399,0.587336,0.273931,0.1711,0.1218,0.2429,"[[0.8735777139663696, 0.8897836804389954]]","[[[0.8735777139663696, 0.2801356613636017], [0...","[[0.8587068915367126, 0.887911856174469]]","[[[0.8587068915367126, 0.29343220591545105], [..."
1,nb_s105046_mpnn_2_IL3,4.861342,89.483418,63.02,2.94,3.526136,85.492289,0.753162,0.4026,0.5049,...,2.438073,0.752653,0.491230,0.1336,0.3481,0.4818,"[[0.9269569516181946, 0.3172776699066162]]","[[[0.9269569516181946, 0.16429194808006287], [...","[[0.9257946014404297, 0.36133983731269836]]","[[[0.9257946014404297, 0.2954653203487396], [0..."
2,nb_s105306_mpnn_2_BHRF,5.242583,89.769324,57.36,2.65,3.632222,88.801675,0.700666,0.4583,0.6721,...,4.563886,0.577668,0.228209,0.1944,0.1246,0.2130,"[[0.9164183139801025, 0.9224730134010315]]","[[[0.9164183139801025, 0.5580517649650574], [0...","[[0.9157121777534485, 0.922823429107666]]","[[[0.9157121777534485, 0.5435625910758972], [0..."
3,nb_s105306_mpnn_3_BHRF,12.129656,78.995566,204.71,9.87,6.875641,61.211013,0.036913,0.0798,0.0185,...,2.527146,0.765890,0.526831,0.3650,0.3400,0.4818,"[[0.9163332581520081, 0.9181700944900513]]","[[[0.9163332581520081, 0.5410971641540527], [0...","[[0.9205971956253052, 0.9215781092643738]]","[[[0.9205971956253052, 0.7132046222686768], [0..."
4,nb_s109706_mpnn_2_BHRF,5.557438,89.234917,31.00,1.86,3.048485,90.788485,0.738247,0.4322,0.7636,...,1.326977,0.842383,0.736508,0.3910,0.6978,0.5874,"[[0.9237253665924072, 0.9229375720024109]]","[[[0.9237253665924072, 0.8435201644897461], [0...","[[0.9260150790214539, 0.9301165342330933]]","[[[0.9260150790214539, 0.8554625511169434], [0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,nb_s949322_mpnn_4_BHRF,5.784341,88.236601,185.07,8.82,3.860000,86.359230,0.628557,0.4430,0.6266,...,1.862179,0.766293,0.593591,0.3306,0.4835,0.4994,"[[0.8663344979286194, 0.9087797403335571]]","[[[0.8663344979286194, 0.255534827709198], [0....","[[0.8703155517578125, 0.9116320013999939]]","[[[0.8703155517578125, 0.48192840814590454], [..."
236,nb_s972445_mpnn_2_BHRF,5.676189,87.873886,86.90,4.21,3.539535,87.190624,0.715671,0.4281,0.5489,...,1.504255,0.898091,0.837154,0.4962,0.6601,0.6329,"[[0.9258394241333008, 0.9077953100204468]]","[[[0.9258394241333008, 0.7158982157707214], [0...","[[0.9366505742073059, 0.9215488433837891]]","[[[0.9366505742073059, 0.8184788227081299], [0..."
237,nb_s975197_mpnn_2_PDL1,8.030759,84.102130,63.04,4.39,3.966071,80.833482,0.305648,0.2114,0.2603,...,2.417347,0.749999,0.461919,0.1710,0.2798,0.4691,"[[0.9257193803787231, 0.8899508714675903]]","[[[0.9257193803787231, 0.613598644733429], [0....","[[0.9261916875839233, 0.9330374002456665]]","[[[0.9261916875839233, 0.7542924880981445], [0..."
238,nb_s983700_mpnn_1_BHRF,7.021542,86.848448,35.15,1.95,5.519444,81.308138,0.430267,0.2746,0.3988,...,1.452703,0.787254,0.656369,0.3502,0.5857,0.5229,"[[0.8850921392440796, 0.9291619658470154]]","[[[0.8850921392440796, 0.7398593425750732], [0...","[[0.8912460207939148, 0.9345695972442627]]","[[[0.8912460207939148, 0.7942801713943481], [0..."


## master df

##### snir


In [82]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df

,source,type,sample,fab,interface,vh/vl,target,target_residues,target_sequence,binder,KD[M],notes
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,1,0.000001,Fab 1E01 + KD[M] approximated from Snir et. al...
...,...,...,...,...,...,...,...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLASSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN


In [83]:
master_df2 = master_df.drop(columns=["interface", "fab", "vh/vl", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,sample,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1,0.000010
1,snir_ab,original,5ifj_abc_cut,1,0.000010
2,snir_ab,original,5ig7_abc_cut,1,0.000100
3,snir_ab,original,5ig7_abc_cut,1,0.000100
4,snir_ab,original,5ijk_acx_cut,1,0.000001
...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,0,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,0,NaN


In [84]:
master_df3 = master_df2.drop_duplicates(keep="last")
master_df3

,source,type,sample,binder,KD[M]
1,snir_ab,original,5ifj_abc_cut,1,0.000010
3,snir_ab,original,5ig7_abc_cut,1,0.000100
5,snir_ab,original,5ijk_acx_cut,1,0.000001
7,snir_ab,alanine_scan,5ifj_abc_cut_A_D111A,0,NaN
9,snir_ab,alanine_scan,5ifj_abc_cut_A_K110A,0,NaN
11,snir_ab,alanine_scan,5ifj_abc_cut_A_S64A,0,NaN
13,snir_ab,alanine_scan,5ifj_abc_cut_A_W52A,0,NaN
15,snir_ab,alanine_scan,5ifj_abc_cut_A_Y66A,0,NaN
17,snir_ab,alanine_scan,5ifj_abc_cut_all_A,0,NaN
19,snir_ab,alanine_scan,5ifj_abc_cut_B_L116A,0,NaN


##### mcmahon / germinal

In [90]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df.head()

,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M]
0,germinal,original,nb_s413501_mpnn_3,QVQLVESGGGLVQPGGSLRLSCAASGSRAIHLKYVGLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN
1,germinal,original,nb_s80903_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASGSLYISNCKTHLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN
2,germinal,original,nb_s185784_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASRWFSNEFHNICLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN
3,germinal,original,nb_s465730_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASSANFSLSGFHHLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN
4,germinal,original,nb_s892360_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASSRFPWEFIRMWLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN


In [91]:
AMBIGUOUS_TARGET= "HSA"   # test

def create_sample_id(row):
    vhh = str(row['vhh'])
    target = str(row['target'])
    residues = str(row['target_residues'])
    
    # Check if target is (ambigous target, meaning the same target has different version --> target residues are needed to be able to identify it, the only one for now is HSA) to apply the specific naming convention
    if target == AMBIGUOUS_TARGET:
        return f"{vhh}_{target}_{residues}"
    else:
        return f"{vhh}_{target}"

# 1. Create the 'sample' column in the master table
master_df['sample'] = master_df.apply(create_sample_id, axis=1)
master_df



,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],sample
0,germinal,original,nb_s413501_mpnn_3,QVQLVESGGGLVQPGGSLRLSCAASGSRAIHLKYVGLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s413501_mpnn_3_PDL1
1,germinal,original,nb_s80903_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASGSLYISNCKTHLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s80903_mpnn_4_PDL1
2,germinal,original,nb_s185784_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASRWFSNEFHNICLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s185784_mpnn_4_PDL1
3,germinal,original,nb_s465730_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASSANFSLSGFHHLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s465730_mpnn_4_PDL1
4,germinal,original,nb_s892360_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASSRFPWEFIRMWLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s892360_mpnn_4_PDL1
...,...,...,...,...,...,...,...,...,...,...
475,germinal,target_shuffle,nb_s276607_mpnn_4,QVQLVESGGGLVQPGGSLRLSCAASGDDVDKWFLDWLGWFRQAPGQ...,IL20,25-176,LKTLNLGSCVIATNLQEIRNGFSEIRGSVQAKDGNIDIRILRRTES...,0,NaN,nb_s276607_mpnn_4_IL20
476,germinal,target_shuffle,nb_s442197_mpnn_3,QVQLVESGGGLVQPGGSLRLSCAASFDLSRCYMISYLGWFRQAPGQ...,PDL1,19-127,FTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNI...,0,NaN,nb_s442197_mpnn_3_PDL1
477,germinal,target_shuffle,nb_s776407_mpnn_2,QVQLVESGGGLVQPGGSLRLSCAASDFFTSKYHWWYLGWFRQAPGQ...,IL3,31-152,SWVNCSNMIDEIITHLKQPPLPLLDFNNLNGEDQDILMENNLRRPN...,0,NaN,nb_s776407_mpnn_2_IL3
478,germinal,target_shuffle,nb_s206651_mpnn_1,QVQLVESGGGLVQPGGSLRLSCAASGMTAAEFKERELGWFRQAPGQ...,IL3,31-152,SWVNCSNMIDEIITHLKQPPLPLLDFNNLNGEDQDILMENNLRRPN...,0,NaN,nb_s206651_mpnn_1_IL3


In [93]:
master_df2 = master_df.drop(columns=["vhh", "vhh_sequence", "target", "target_residues", "target_sequence"])
master_df2

,source,type,binder,KD[M],sample
0,germinal,original,0,NaN,nb_s413501_mpnn_3_PDL1
1,germinal,original,0,NaN,nb_s80903_mpnn_4_PDL1
2,germinal,original,0,NaN,nb_s185784_mpnn_4_PDL1
3,germinal,original,0,NaN,nb_s465730_mpnn_4_PDL1
4,germinal,original,0,NaN,nb_s892360_mpnn_4_PDL1
...,...,...,...,...,...
475,germinal,target_shuffle,0,NaN,nb_s276607_mpnn_4_IL20
476,germinal,target_shuffle,0,NaN,nb_s442197_mpnn_3_PDL1
477,germinal,target_shuffle,0,NaN,nb_s776407_mpnn_2_IL3
478,germinal,target_shuffle,0,NaN,nb_s206651_mpnn_1_IL3


### merge

In [97]:
scores_df = pd.merge(master_df2, df_2if, on=['sample'], how='inner')
#df = pd.merge(master_df2, preds_df, on=['sample'], how='inner')
# Display the first few rows to verify
print(scores_df[['sample', 'af3_plddt']])

# Save the merged result
scores_df.to_csv(OUTPUT_DIR /'merged_germinal.csv', index=False)

                     sample  af3_plddt
0    nb_s413501_mpnn_3_PDL1  86.742610
1     nb_s80903_mpnn_4_PDL1  87.377710
2    nb_s185784_mpnn_4_PDL1  87.466883
3    nb_s465730_mpnn_4_PDL1  85.815865
4    nb_s892360_mpnn_4_PDL1  86.163597
..                      ...        ...
235  nb_s276607_mpnn_4_BHRF  90.501328
236  nb_s442197_mpnn_3_BHRF  82.902422
237  nb_s776407_mpnn_2_BHRF  80.288672
238  nb_s206651_mpnn_1_BHRF  89.919324
239  nb_s972445_mpnn_2_BHRF  87.873886

[240 rows x 2 columns]
